In [ ]:
import os
import rasterio
import numpy as np
import folium
from rasterio.warp import transform_bounds
from branca.colormap import linear
import geopandas as gpd


In [17]:
def tif_to_folium(tif_path, output_html):
    with rasterio.open(tif_path) as src:
        data = src.read(1).astype(float)
        bounds = src.bounds
        crs = src.crs

        nodata = src.nodata
        if nodata is not None:
            data[data == nodata] = np.nan

        # Reproject bounds if needed
        if crs and crs.to_string() != "EPSG:4326":
            bounds = transform_bounds(
                crs, "EPSG:4326",
                bounds.left, bounds.bottom,
                bounds.right, bounds.top
            )

        min_val = np.nanmin(data)
        max_val = np.nanmax(data)

        print(min_val,max_val)
        # 🔴 CRITICAL FIX
        if not np.isfinite(min_val) or not np.isfinite(max_val):
            print(f"Skipping {tif_path}: invalid raster")
            return

        if min_val == max_val:
            max_val = min_val + 1e-6

    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2

    m = folium.Map(location=[center_lat, center_lon], zoom_start=6)

    colormap = linear.YlOrRd_09.scale(min_val, max_val)
    # colormap.caption = os.path.basename(tif_path)

    folium.raster_layers.ImageOverlay(
        image=data,
        bounds=[[bounds[1], bounds[0]], [bounds[3], bounds[2]]],
        opacity=0.7,
        colormap=lambda x: colormap(x)
    ).add_to(m)

    colormap.add_to(m)
    m.save(output_html)


In [18]:
import plotly.graph_objects as go
import plotly.express as px
import rasterio
import numpy as np
import plotly.express as px
import os

arcgis_colorscale = [
    [0.0,  "#f7f4f2"],  # very low (almost white)
    [0.25, "#e0d7cf"],
    [0.5,  "#c4b5a7"],  # medium
    [0.75, "#9c7f6a"],
    [1.0,  "#5c4033"]   # high (dark brown)
]

def visualize_and_save(tif_path, output_html_path):
    """Read a TIF file, visualize it using Plotly, and save as HTML."""
    with rasterio.open(tif_path) as src:
        data = src.read(1)  # Read the first band
        # Mask no-data values
        data = np.ma.masked_equal(data, src.nodata)
        fig = px.imshow(
            data,
            color_continuous_scale='Viridis',
            labels=dict(color="Value"),
            title=os.path.basename(tif_path)
        )
        fig.write_html(output_html_path)
        print(f"Saved: {output_html_path}")

def visualize_and_save(tif_path, output_html_path):
    """Read a TIF file, visualize it using Plotly with ArcGIS-like colors, and save as HTML."""
    
    with rasterio.open(tif_path) as src:
        data = src.read(1)
        data = np.ma.masked_equal(data, src.nodata)

        fig = px.imshow(
            data,
            color_continuous_scale=arcgis_colorscale,
            labels=dict(color="Value"),
            title=os.path.basename(tif_path)
        )
        # fig = px.imshow(
        #     data,
        #     color_continuous_scale=arcgis_colorscale,
        #     range_color=(0, 25),
        #     labels=dict(color="Value"),
        #     title=os.path.basename(tif_path),
        # )
        fig.update_layout(
            coloraxis_colorbar=dict(
                title="Legend",
                thickness=20,
                len=0.7
            )
        )

        fig.write_html(output_html_path)
        print(f"Saved: {output_html_path}")

def visualize_and_save(tif_path, output_html_path, arcgis_colorscale = None, range_color = None):
    """Read a TIF file, visualize it using Plotly with ArcGIS-like colors, and save as HTML."""
    
    with rasterio.open(tif_path) as src:
        data = src.read(1)
        data = np.ma.masked_equal(data, src.nodata)

        fig = px.imshow(
            data,
            color_continuous_scale=arcgis_colorscale,
            range_color=range_color,
            labels=dict(color="Value"),
            title=os.path.basename(tif_path),
        )

        fig.update_layout(
            paper_bgcolor="black",   # outside background
            plot_bgcolor="black",    # image background
            font=dict(color="white"),
            coloraxis_colorbar=dict(
                title="Legend",
                thickness=20,
                len=0.7,
                tickcolor="white",
                title_font=dict(color="white")
            )
        )

        fig.write_html(output_html_path)
        print(f"Saved: {output_html_path}")


In [19]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np
from rasterio.mask import mask

def mask_raster_to_shape(
    raster_path,
    shape_path,
    output_path
):
    gdf = gpd.read_file(shape_path)

    with rasterio.open(raster_path) as src:
        gdf = gdf.to_crs(src.crs)

        out_image, out_transform = mask(
            src,
            gdf.geometry,
            crop=False,       # keep original extent
            filled=True,
            nodata=np.nan
        )

        out_meta = src.meta.copy()
        out_meta.update({
            "nodata": np.nan
        })

    with rasterio.open(output_path, "w", **out_meta) as dst:
        dst.write(out_image)



In [ ]:
def process_directory(input_root, output_root=None):
    base_name = os.path.basename(input_root.rstrip("/"))
    if output_root is None:
        output_root = f"html_maps_{base_name}"

    for root, _, files in os.walk(input_root):
        for file in files:
            if file.lower().endswith(".tif"):
                tif_path = os.path.join(root, file)

                # Preserve subfolder structure
                rel_path = os.path.relpath(root, input_root)
                output_dir = os.path.join(output_root, rel_path)
                os.makedirs(output_dir, exist_ok=True)

                html_name = os.path.splitext(file)[0] + ".html"
                output_html = os.path.join(output_dir, html_name)

                print(f"Creating map: {output_html}")
                # tif_to_folium(tif_path, output_html)
                mask_raster_to_shape(
                    raster_path = tif_path,
                    shape_path = 'external_data/algeria.geojson',
                    output_path = tif_path)

                visualize_and_save(tif_path, output_html)



In [21]:
input_folders = [
    # "mlp_maps",
    # "maps_6",
    # "maps_2",
    # "cnn_basic_maps"
    #  "outputs_unceratinty_mtcnn"
    # "sand_uncer_without_outliers"
    # "outputs_uncertainty_mlp"
    "maps/RandomForest"
]

for folder in input_folders:
    process_directory(folder, folder)


Creating map: maps/RandomForest\.\rf_A%_map.html
Saved: maps/RandomForest\.\rf_A%_map.html
Creating map: maps/RandomForest\.\rf_CEC_map.html
Saved: maps/RandomForest\.\rf_CEC_map.html
Creating map: maps/RandomForest\.\rf_CE_map.html
Saved: maps/RandomForest\.\rf_CE_map.html
Creating map: maps/RandomForest\.\rf_L%_map.html
Saved: maps/RandomForest\.\rf_L%_map.html
Creating map: maps/RandomForest\.\rf_OM_map.html
Saved: maps/RandomForest\.\rf_OM_map.html
Creating map: maps/RandomForest\.\rf_pH_map.html
Saved: maps/RandomForest\.\rf_pH_map.html
Creating map: maps/RandomForest\.\rf_S%_map.html
Saved: maps/RandomForest\.\rf_S%_map.html
